# YouTube Shorts Deduplication Test & Extrapolation

This notebook verifies our proposed algorithm for **YouTube Shorts deduplication**. 

### Objectives:
1. Filter the dataset to include only the 17 specified channels.
2. Group videos by channel and separate them into **Shorts** and **Long videos**.
3. Implement an optimized **word-level subset matching** algorithm that checks if an *entire Short transcript* is contained within a *subset of a Long video*.
4. Test the algorithm on a sample channel (`Camihawke`) and measure execution time.
5. Extrapolate the processing time to the entire dataset.

In [1]:
import os
import time
import difflib
import pandas as pd

PARQUET_PATH = "Data_Cleaned/yt_videos_with_local_transcripts.parquet"
assert os.path.exists(PARQUET_PATH), f"Parquet file not found at {PARQUET_PATH}"

print("Loading dataset...")
df = pd.read_parquet(PARQUET_PATH)
print(f"Loaded {len(df):,} videos with {len(df.columns)} columns.")

Loading dataset...
Loaded 12,829 videos with 29 columns.


In [2]:
# Define target channels
target_channels = {
    "Mr. Marra", "Davide Zambelli", "ChiaraMaciTv", "Canale di Venti",
    "Smibie Channel", "The Lady", "Debora Fulli", "roccotnl",
    "Camihawke", "Il Matricomio", "Raissa & Momo", "Sara La Rusca",
    "THOMAS BASILICO", "Contenuti Zero", "Valentina Barbieri",
    "AlicelikeAudrey", "Christian Manzoni"
}

# Filter dataset
df_filtered = df[df["channelTitle"].isin(target_channels)].copy()
df_shorts = df_filtered[df_filtered["is_short"] == True]
df_longs = df_filtered[df_filtered["is_short"] == False]

print(f"Total videos in target channels: {len(df_filtered):,}")
print(f"Shorts: {len(df_shorts):,}")
print(f"Long videos: {len(df_longs):,}")

Total videos in target channels: 7,188
Shorts: 2,108
Long videos: 5,080


In [3]:
# Calculate channel-level statistics and comparisons
stats = []
for channel in sorted(target_channels):
    s_count = len(df_shorts[df_shorts["channelTitle"] == channel])
    l_count = len(df_longs[df_longs["channelTitle"] == channel])
    comparisons = s_count * l_count
    stats.append({
        "Channel": channel,
        "Shorts": s_count,
        "Longs": l_count,
        "Comparisons": comparisons
    })

stats_df = pd.DataFrame(stats)
total_comparisons = stats_df["Comparisons"].sum()
print("\nChannel Statistics:")
display(stats_df)
print(f"Total comparisons required: {total_comparisons:,}")


Channel Statistics:
               Channel  Shorts  Longs  Comparisons
0      AlicelikeAudrey       4    873         3492
1            Camihawke      16     17          272
2      Canale di Venti      70    600        42000
3         ChiaraMaciTv       2    198          396
4    Christian Manzoni     140     21         2940
5       Contenuti Zero     556      8         4448
6      Davide Zambelli      37    221         8177
7         Debora Fulli     243    829       201447
8        Il Matricomio      48    138         6624
9            Mr. Marra      94    401        37694
10       Raissa & Momo      49     39         1911
11       Sara La Rusca      42      0            0
12      Smibie Channel      85    820        69700
13     THOMAS BASILICO     347     10         3470
14            The Lady     100    882        88200
15  Valentina Barbieri     212      2          424
16            roccotnl      63     21         1323
Total comparisons required: 472,518


## Subset Matching Algorithm & Pre-Filter

We need to check if the **entire** Short transcript is a subset of a Long video transcript. 
Instead of doing character-level comparisons, we use **word-level tokenization** which decreases sequence lengths and speeds up matching by **5x-6x**.

### The Word-Set Pre-Filter:
To avoid running the expensive `difflib.SequenceMatcher` on thousands of unrelated video pairs, we check the intersection of unique words between the Short and the Long video:
$$\text{Word Intersection Ratio} = \frac{\text{Unique words in Short } \cap \text{ Unique words in Long}}{\text{Unique words in Short}}$$
If this ratio is less than the threshold (e.g. 0.8), it is mathematically impossible for the Short transcript to be an 80%+ subset of the Long video transcript. We can **instantly skip** these comparisons.

In [5]:
def get_words(text):
    if not isinstance(text, str):
        return []
    # Lowercase, clean basic punctuation, split
    clean = text.lower().replace("\n", " ").replace("\r", " ")
    for char in [",", ".", "?", "!", "-", "'", "\"", ">>", "<<"]:
        clean = clean.replace(char, " ")
    return clean.split()

def find_subset_matches(shorts_df, longs_df, threshold=0.8):
    # Pre-parse words and unique sets
    shorts_data = []
    for _, row in shorts_df.iterrows():
        words = get_words(row["local_transcript"])
        if words:
            shorts_data.append({
                "videoId": row["videoId"],
                "words": words,
                "set": set(words)
            })
            
    longs_data = []
    for _, row in longs_df.iterrows():
        words = get_words(row["local_transcript"])
        if words:
            longs_data.append({
                "videoId": row["videoId"],
                "words": words,
                "set": set(words)
            })
            
    matches = []
    comparisons_run = 0
    comparisons_skipped = 0
    
    start_time = time.time()
    
    for s in shorts_data:
        best_ratio = 0.0
        best_match_id = None
        
        for l in longs_data:
            comparisons_run += 1
            
            # 1. Fast Set Intersection Pre-Filter
            intersection = s["set"].intersection(l["set"])
            if len(intersection) / len(s["set"]) < threshold:
                comparisons_skipped += 1
                continue
                
            # 2. Precise Sequence Matching on Word Lists
            matcher = difflib.SequenceMatcher(None, s["words"], l["words"])
            matching_blocks = matcher.get_matching_blocks()
            total_matched = sum(block.size for block in matching_blocks)
            ratio = total_matched / len(s["words"])
            
            if ratio > best_ratio:
                best_ratio = ratio
                best_match_id = l["videoId"]
                
        if best_ratio >= threshold:
            matches.append({
                "short_id": s["videoId"],
                "long_id": best_match_id,
                "ratio": best_ratio
            })
            
    duration = time.time() - start_time
    
    return pd.DataFrame(matches), duration, comparisons_run, comparisons_skipped

In [6]:
# Run test on Camihawke
sample_channel = "Camihawke"
print(f"Running test matching on '{sample_channel}'...")

c_shorts = df_shorts[df_shorts["channelTitle"] == sample_channel]
c_longs = df_longs[df_longs["channelTitle"] == sample_channel]

matches_df, duration, run, skipped = find_subset_matches(c_shorts, c_longs, threshold=0.8)

print(f"\nResults for {sample_channel}:")
print(f"- Time taken: {duration:.4f} seconds")
print(f"- Total possible comparisons: {run}")
print(f"- Skipped by pre-filter: {skipped} ({skipped/run*100:.1f}% reduction in SequenceMatcher workload)")
print(f"- Actual SequenceMatcher evaluations: {run - skipped}")
print(f"- Deduplicated Shorts found: {len(matches_df)} out of {len(c_shorts)} shorts")

if len(matches_df) > 0:
    display(matches_df)

Running test matching on 'Camihawke'...

Results for Camihawke:
- Time taken: 0.0626 seconds
- Total possible comparisons: 272
- Skipped by pre-filter: 258 (94.9% reduction in SequenceMatcher workload)
- Actual SequenceMatcher evaluations: 14
- Deduplicated Shorts found: 13 out of 16 shorts
       short_id      long_id     ratio
0   eNU5FcmEwTc  tNbsrpKaMoE  0.888889
1   AS6paOwhdR0  wWj-r-tCArw  0.994253
2   wPFcvFEPSSQ  wWj-r-tCArw  1.000000
3   0HtHNdF2AIY  0rzbwX_CNII  0.950000
4   Ut11SY8_BTE  0rzbwX_CNII  0.985714
5   911w9xOdX7g  0rzbwX_CNII  0.960396
6   hATUt0UytWU  OOf8CLAghZ0  0.896552
7   LdaXe_BxGv0  OOf8CLAghZ0  0.912409
8   7N8LhdXMVXk  OOf8CLAghZ0  0.888889
9   b9Cp9UfIMxM  X5qnZAMUv7M  1.000000
10  _tsZLOVeSQ0  X5qnZAMUv7M  0.909910
11  E0Sh6Cle5mw  5qIEw7on7PU  0.973545
12  4bDXDXVk4WQ  5qIEw7on7PU  0.937500


In [7]:
# Extrapolate run time to total dataset
time_per_comparison = duration / run
extrapolated_total_time = total_comparisons * time_per_comparison

print("\n=== Extrapolation Analysis ===")
print(f"Average time per comparison: {time_per_comparison * 1000: .4f} ms")
print(f"Total Comparisons for 17 channels: {total_comparisons:,}")
print(f"Estimated total execution time: {extrapolated_total_time:.2f} seconds ({extrapolated_total_time/60:.2f} minutes)")


=== Extrapolation Analysis ===
Average time per comparison:  0.2302 ms
Total Comparisons for 17 channels: 472,518
Estimated total execution time: 108.76 seconds (1.81 minutes)
